In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# Load data
data = pd.read_csv("../DATASET/Plant_Parameters.csv")

# Separate features and target
target = data['Moisture']
features = data.drop('Moisture', axis=1)

# Encode categorical variable if needed
le = LabelEncoder()
features['Plant Type'] = le.fit_transform(features['Plant Type'])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# we must apply the scaling to the test set as well that we are computing for the training set
X_test_scaled = scaler.transform(X_test)


# Define the XGBoost regressor
xgb_reg = xgb.XGBRegressor(eta = 0.01, gamma = 70, learning_rate = 0.09, max_depth = 4, n_estimators = 100)

# Train the model
xgb_reg.fit(X_train_scaled, y_train)

# Make predictions
y_pred = xgb_reg.predict(X_test_scaled)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R-squared Score:", r2)

Mean Squared Error: 10.865180412479075
R-squared Score: 0.8252472318332342


In [5]:
import joblib

# Save the model to a file
joblib.dump(xgb_reg, 'model.pkl')

print("XGBoost model saved as model.pkl")

XGBoost model saved as model.pkl


In [11]:
import pickle

# Load the XGBoost model from the pickle file
with open("model.pkl", "rb") as f:
    xgb_reg = pickle.load(f)

# Assuming you have a sample input data stored in a dictionary
sample_input = {
    'pH': 6.5,
    'Soil EC': 0.5,
    'Phosphorus': 20,
    'Potassium': 40,
    'Urea': 30,
    'T.S.P': 40,
    'M.O.P': 30,
    'Moisture': 25,  # This value will not be used for prediction
    'Temperature': 25,
    'Plant Type': 1
}

# Extract features from the input data (excluding 'Moisture')
features = [sample_input[col] for col in sample_input if col != 'Moisture']

# Make prediction using the loaded model
prediction = xgb_reg.predict([features])[0]

print("Predicted Moisture:", prediction)


Predicted Moisture: 65.27302
